# 29. 골든 확장·카드 보강·공동 DB 적재·UI 개선 작업 기록 (2026-11-06 ~ 11-07)

> 작성: 유종혁(예방·법령). 진행 중 Codex 검토를 병행했고, 아래 숫자는 저장소 기록(`data/specs/golden/prevention_legal.md` 7-17~7-26절,
> `docs/progress.md`, `notebooks/prevention_legal/일정관리_prevention_legal.md`)과 같다. 세세한 로그는 그 파일들에 있고 여기에는 흐름과 **특이점**만 남긴다.
> 코드 셀은 모두 **읽기 전용**이다(공동 DB에 쓰지 않고, `.env` 값·접속 문자열을 출력하지 않는다). 다만 `run()`·`connect()`는 공동 DB에 **연결**하므로 실행하려면 `.env`의 접속 설정과 DB 상태가 필요하다. §3의 셀은 Git 제외 파일 `data/interim/prevention_legal/blocks.jsonl`(로컬 블록)도 필요해서 다른 PC에서는 따로 준비해야 한다.

| 구분 | 내용 | 결과 |
|---|---|---|
| 골든 | 15개 세트(회귀 7 + 독립 8, 5유형 3개씩) 실행 전 고정 후 1회 실행 | 회귀 **7/7**, 독립 **2/8** (합산하지 않음) |
| 카드 | Q06 비교 카드 쌍, Q28 거부 카드, 법 제12조 ① 카드(S-L01·Q09), Q10 카드 신고 대상·주체 수정, **벌칙 카드 3개(제81조·제79조의4·제80조)** | 15개 세트·22개 질문 결과 변화 없음 |
| 공동 DB | 홍역 94쪽 1개, 벌칙 조문 12개 적재 (승인 후) | Chunk 1730 → 1731 → 1743 (HAS_CHUNK도 같은 수만큼 증가). 확인한 범위(지정 질문 결과·문서 필터 검색)에서 기존 결과 변화 없음 |
| UI | 일반/개발자 화면 분리, 저장한 답변, 근거 문서, 문의처 안내 철회 | AppTest 확인(브라우저 시각 확인은 사용자) |
| 시험 | 전체 `pytest tests` | 벌칙 카드 추가 전 319, **추가 후 372 통과**(9 건너뜀) |

In [1]:
# 공통 준비: 프로젝트 소스 경로만 추가한다. .env 는 neo4j_conn 이 읽고, 이 노트북은 값을 출력하지 않는다.
import sys, re, logging
logging.getLogger("neo4j").setLevel(logging.ERROR)    # 전문 검색 인덱스 함수의 'deprecated' 안내(동작에는 영향 없음)가 출력을 가리지 않게 한다
sys.path.insert(0, "../../src")
from rag.prevention_legal.run import run
from rag.prevention_legal.retrieve import fetch_chunks

ALLOWED = ["kdca_measles_2026", "kdca_diagnosis_reporting_2026", "moleg_infectious_disease_act_21521",
           "kdca_hfmd_2026", "kdca_vaccine_preventable_2026"]   # 골든 평가에 쓴 허용 문서 5개

## 1. 골든 15개 세트와 평가 규칙

- **회귀(R1~R7):** 답 카드를 허용한다. 질문을 보고 만든 카드이므로 "새 질문 대응력"이 아니라 "알려진 질문이 유지되는지"를 본다.
- **독립(I1~I8):** 카드를 **만들지 않는다**. 결과가 근거 부족(INSUFFICIENT_EVIDENCE)이어도 그대로 보고한다.
- 기대 상태와 질문은 실행 **전에** 명세 7-22절에 고정했고, 실행 뒤 바꾸지 않았다. 비교 문항 채점 규칙: 두 문서 모두에 질문이 묻는 조건이 있으면 ANSWERED, 한쪽만 있으면 PARTIAL.
- **재사용 주의:** 독립 8개 중 7개(I1~I3, I5~I8)는 이미 한 번 실행한 독립 문항(I-1~I-4, I-6, J-3, J-4)을 다시 쓴 것이다. 같은 결과가 다시 나오므로 새 일반화 근거가 아니다. 새로 만든 독립 문항은 I4 하나다.
- **두 점수를 합치지 않는다.** 독립 일치 2건은 I7·I8로 둘 다 기대가 "근거 없음"인 질문이라 근거 있는 답변의 성공이 아니다.

아래 셀은 15개를 다시 실행(읽기 전용)해서 회귀·독립을 따로 센다.

In [2]:
# 15개 세트 재실행(읽기 전용). 질문·기대 상태는 명세 7-22절과 같다.
SET = [
 ("R1","회귀","질병","수족구병에 걸린 학생은 학교에 계속 나와도 되나요?","ANSWERED"),
 ("R2","회귀","문서 비교","대학교에서 홍역 환자가 발생했을 때 학교 담당자의 역할은 초·중·고와 어떻게 다른가요?","ANSWERED"),
 ("R3","회귀","학교 행동","학교 대상 임시예방접종은 어떤 절차로 시행하나요?","ANSWERED"),
 ("R4","회귀","학교 행동","수족구병으로 등교를 중지한 학생은 언제부터 다시 등교할 수 있나요?","ANSWERED"),
 ("R5","회귀","신고","홍역 환자 또는 의사환자의 발생 신고 시기는 언제인가요?","ANSWERED"),
 ("R6","회귀","신고","신고를 받은 보건소장은 그 다음에 어떤 보고를 하나요?","ANSWERED"),
 ("R7","회귀","근거 부족","학교가 신고를 늦게 하면 어떤 처벌을 받는지 확정해서 알려 주세요.","INSUFFICIENT_EVIDENCE"),
 ("I1","독립","질병","수족구병에서 완치 후 병원체 배출 가능성을 최소 얼마 동안 고려해야 하나요?","ANSWERED"),
 ("I2","독립","질병","성홍열의 신고 범위와 각 대상의 진단기준은 무엇인가요?","ANSWERED"),
 ("I3","독립","문서 비교","2026 홍역 대응 지침과 2026 수족구 지침의 등교 중지 안내 기간은 같은가요?","PARTIAL"),
 ("I4","독립","문서 비교","홍역 지침의 등교 중지 기준(발진 후 4일)과 수족구 지침의 복귀 기준(발열 없고 수포가 마른 후)은 같은 기준인가요?","ANSWERED"),
 ("I5","독립","학교 행동","수족구병에서 회복된 학생은 복귀 후 학교에서 어떤 위생 수칙을 지켜야 하나요?","ANSWERED"),
 ("I6","독립","신고","제1급감염병 환자를 보고받은 의료기관장은 언제까지 신고해야 하나요?","ANSWERED"),
 ("I7","독립","근거 부족","제공된 자료에 성홍열 접촉자의 예방 항생제 투여 기준이 있나요?","INSUFFICIENT_EVIDENCE"),
 ("I8","독립","근거 부족","홍역 확진 학생의 학교 복귀를 허가하는 기관은 어디인가요?","INSUFFICIENT_EVIDENCE"),
]
compact = lambda t: re.sub(r"[\s*\[\]]", "", t)       # 공백·강조 표시 차이를 무시한 비교용
score = {"회귀": [0, 0], "독립": [0, 0]}
for no, kind, typ, q, expected in SET:
    out = run({"question": q, "region": "NATIONAL", "allowed_document_ids": ALLOWED})
    cids = {c["chunk_id"] for c in out["citations"]}
    db = fetch_chunks(list(cids)) if cids else {}
    # 인용 무결성: Finding 이 가리키는 청크가 모두 Citation 에 있고, 발췌가 DB 청크 원문에 들어 있는지
    # (인용·Finding 이 하나도 없는 문항은 all() 이 빈 조건에서 자명하게 True 가 되므로, 그 줄의 True 는 검사한 것이 없다는 뜻이다)
    ok_link = all(set(f["chunk_ids"]) <= cids for f in out["findings"])
    ok_text = all(c["chunk_id"] in db and compact(c["excerpt"]) in compact(db[c["chunk_id"]]["text"]) for c in out["citations"])
    match = out["status"] == expected
    score[kind][0] += match; score[kind][1] += 1
    print(f"{no} {kind} {typ:6s} 기대 {expected:21s} 실제 {out['status']:21s} {'일치' if match else '불일치'} | 인용 {len(cids)} 연결 {ok_link} 원문 {ok_text}")
print("회귀", score["회귀"], "독립", score["독립"], "(합산하지 않음)")

R1 회귀 질병     기대 ANSWERED              실제 ANSWERED              일치 | 인용 1 연결 True 원문 True


R2 회귀 문서 비교  기대 ANSWERED              실제 ANSWERED              일치 | 인용 2 연결 True 원문 True


R3 회귀 학교 행동  기대 ANSWERED              실제 ANSWERED              일치 | 인용 3 연결 True 원문 True


R4 회귀 학교 행동  기대 ANSWERED              실제 ANSWERED              일치 | 인용 1 연결 True 원문 True


R5 회귀 신고     기대 ANSWERED              실제 ANSWERED              일치 | 인용 1 연결 True 원문 True


R6 회귀 신고     기대 ANSWERED              실제 ANSWERED              일치 | 인용 1 연결 True 원문 True
R7 회귀 근거 부족  기대 INSUFFICIENT_EVIDENCE 실제 INSUFFICIENT_EVIDENCE 일치 | 인용 0 연결 True 원문 True


... (4 outputs left out)

I5 독립 학교 행동  기대 ANSWERED              실제 INSUFFICIENT_EVIDENCE 불일치 | 인용 0 연결 True 원문 True


I6 독립 신고     기대 ANSWERED              실제 INSUFFICIENT_EVIDENCE 불일치 | 인용 0 연결 True 원문 True


I7 독립 근거 부족  기대 INSUFFICIENT_EVIDENCE 실제 INSUFFICIENT_EVIDENCE 일치 | 인용 0 연결 True 원문 True


I8 독립 근거 부족  기대 INSUFFICIENT_EVIDENCE 실제 INSUFFICIENT_EVIDENCE 일치 | 인용 0 연결 True 원문 True
회귀 [7, 7] 독립 [2, 8] (합산하지 않음)


## 2. 카드 변경과 특이점

| 변경 | 내용 | 특이점 |
|---|---|---|
| Q06 비교 카드 쌍 | 대학교(94쪽 b01)와 초·중·고(93쪽 b05) 카드가 **둘 다** 검색될 때만 답한다 | 한쪽만 나오면 비교 답이 아니므로 둘 다 뺀다. 94쪽 블록은 공동 DB에 새로 적재(§3) |
| Q28 거부 카드 | 처벌+지연 질문은 검색 전에 인용 없이 INSUFFICIENT_EVIDENCE | 처음엔 "근거 블록에 벌칙 조문이 없다"고 답했으나 **벌칙 조문 적재 뒤 사실이 아니게 되어** "처벌은 신고 의무자·위반 사실에 따라 달라 확정할 수 없다"로 고침 |
| 법 제12조 ① 카드 | 승우님 요청(학교 골든 S-L01). S-L01·Q09 → **PARTIAL** | 학교 골든은 ANSWERED를 기대했지만 "학교장"↔법의 "관리인, 경영자 또는 대표자" 대응과 복지부령 장소·감염병이 확정되지 않아 PARTIAL. **질문을 보고 만든 회귀 카드**라 독립 평가에는 쓰지 않음 |
| Q10 카드 수정 | 신고 대상·주체를 법 원문대로 수정 | Codex 검토에서 **내 카드가 법의 범위를 넓혀 말하고 있음**이 드러남. 제12조 ①은 "제1급~제3급감염병 중 보건복지부령으로 정하는 감염병", 주체는 "학교 등 보건복지부령으로 정하는 장소의 관리인, 경영자 또는 대표자". 상태(PARTIAL)와 인용 청크는 그대로 |
| 벌칙 카드 3개 | 승우님 요청. 제81조 제3·4호(제12조 ① 신고를 게을리한 자, 200만원 이하), 제79조의4 제1호(제1·2급 신고 위반 의사 등, 500만원 이하), 제80조 제1호(제3·4급, 300만원 이하). 모두 **PARTIAL**, 의사 카드 2개는 둘 다 검색될 때만 답함 | 답을 "처한다"가 아니라 "**처할 수 있습니다, 요건을 충족하는 경우의 조문이며 특정 학교·사람이 해당하는지는 판단하지 않았다**"로 씀(Codex 지적). 단독 '벌' 글자 조건은 '벌써'에 걸려 **벌칙 낱말 묶음 + 주체 낱말 묶음**으로 바꿈. Q28 거부 카드는 처벌의 **'확정' 요구**가 있을 때만 거부하도록 좁힘 |
| 경고 문구 | 카드 없음 / 카드 문서가 허용 목록에 없음 / 카드 블록이 검색되지 않음 세 경우를 모두 말함 | 이전 문구 "고정 샘플 범위 밖"은 현재 동작과 달랐음 |
| Q10 기대 상태 | 이전 9개 평가에서 기대 ANSWERED, 실제 PARTIAL | **기대를 결과에 맞춰 바꾸지 않았다**(사용자 결정). 불일치로 보고. 15개 세트에는 Q10이 없음 |
| I-5 | 기존 Q08 카드가 질문에 반응 | 독립성이 오염되어 독립 집계에서 제외(원 기록 1/6 → 유효 1/5) |

In [3]:
# 벌칙 카드 동작 확인(읽기 전용). 법령 Tool 단독 호출이며 공동 DB 에는 쓰지 않는다.
from rag.prevention_legal.tool import legal_search
LAW_DOCS = ["moleg_infectious_disease_act_21521", "kdca_diagnosis_reporting_2026", "kdca_measles_2026"]
QS = [
    "홍역 환자 발생을 보건소에 신고하지 않으면 학교는 어떤 벌칙을 받나요?",     # 학교 신고 → 제81조 (PARTIAL)
    "신고를 게을리한 경우의 벌칙은 무엇인가요?",                               # 제81조 (PARTIAL)
    "의사가 감염병 환자를 신고하지 않으면 어떤 벌칙을 받나요?",                 # 제79조의4 + 제80조 함께 (PARTIAL)
    "학교가 신고를 늦게 하면 어떤 처벌을 받나요?",                             # 확정 요구 없음 → 조건부 답 (PARTIAL)
    "학교가 신고를 늦게 하면 어떤 처벌을 받는지 확정해서 알려 주세요.",           # 확정 요구 → 거부 (근거 부족)
    "학교에 신고가 벌써 접수됐나요?",                                         # 벌칙 낱말 아님 → 발동하지 않음
    "결핵 환자를 신고하지 않으면 벌금이 있나요?",                              # 결핵은 다른 법 → 답하지 않음
    "신고를 하지 않으면 과태료는 얼마인가요?",                                 # 과태료는 이 카드가 다루지 않음
]
for q in QS:
    r = legal_search({"question": q, "task": "확인", "region": "GYEONGGI", "allowed_document_ids": LAW_DOCS})
    cites = [c["chunk_id"].split(":", 1)[1] for c in r["citations"]]
    print(f"{r['status']:12s} {cites} | {q}")

PARTIAL      ['p0057:b02'] | 홍역 환자 발생을 보건소에 신고하지 않으면 학교는 어떤 벌칙을 받나요?


PARTIAL      ['p0057:b02'] | 신고를 게을리한 경우의 벌칙은 무엇인가요?


PARTIAL      ['p0056:b04', 'p0056:b05'] | 의사가 감염병 환자를 신고하지 않으면 어떤 벌칙을 받나요?


PARTIAL      ['p0057:b02'] | 학교가 신고를 늦게 하면 어떤 처벌을 받나요?
NO_EVIDENCE  [] | 학교가 신고를 늦게 하면 어떤 처벌을 받는지 확정해서 알려 주세요.


NO_EVIDENCE  [] | 학교에 신고가 벌써 접수됐나요?


NO_EVIDENCE  [] | 결핵 환자를 신고하지 않으면 벌금이 있나요?


NO_EVIDENCE  [] | 신고를 하지 않으면 과태료는 얼마인가요?


**한계:** 낱말 일치 게이트라 표현이 다르면(예: "보고를 누락한 의사") 놓칠 수 있다. 의사 카드는 급수 두 조문 중 한쪽만 검색돼도 둘 다 빠진다. 이 카드도 질문을 보고 만든 **회귀 카드**라 독립 평가에는 쓰지 않는다. 메인 그래프에서는 승우님 `check_facts`가 벌칙 청크를 따로 쓰므로 이 카드와는 별개다. 명세 7-27절.

## 3. 공동 DB 쓰기 기록 (모두 사용자 승인 뒤, 새 ID만 CREATE)

| 작업 | 변경 | 확인한 것 |
|---|---|---|
| 홍역 94쪽 `kdca_measles_2026:p0094:b01` | Chunk 1730 → 1731 | 기존 22개 질문 결과 변화 없음, 학교 문서 필터 검색에서 홍역 블록 0건 (HAS_CHUNK도 +1) |
| 벌칙 조문 12개 (법 11 + 진단·신고기준 지침 27쪽 1) | Chunk 1731 → 1743 | DB 텍스트·쪽·절·문서가 로컬 블록과 같음, HAS_CHUNK 외 관계 0, 임베딩 1536차원, 15개 세트·22개 질문 결과 변화 없음(확인한 범위) |

적재 함수는 `confirm=True` 가 있어야 실행되고, 같은 ID·문서 존재 여부를 트랜잭션 안에서 검사해 어긋나면 되돌린다.
아래 셀은 **읽기 전용**으로 현재 DB 상태를 확인한다.

In [4]:
# 읽기 전용 확인: 개수, 94쪽 1개, 벌칙 12개의 존재·본문 일치. 쓰기 쿼리는 없다.
import json
from rag.prevention_legal.neo4j_conn import connect
from rag.prevention_legal.load_graph import PENALTY_CHUNK_IDS, PAGE94_CHUNK_ID

local = {}
for line in open("../../data/interim/prevention_legal/blocks.jsonl", encoding="utf-8"):
    b = json.loads(line); local[b["chunk_id"]] = b          # 로컬 블록(Git 제외 파일)

driver = connect()
with driver.session() as s:
    one = lambda cy, **p: s.run(cy, **p).single()[0]
    print("Chunk", one("MATCH (c:Chunk) RETURN count(c)"), "| Document", one("MATCH (d:Document) RETURN count(d)"),
          "| HAS_CHUNK", one("MATCH ()-[r:HAS_CHUNK]->() RETURN count(r)"))
    rows = s.run("MATCH (d:Document)-[:HAS_CHUNK]->(c:Chunk) WHERE c.chunk_id IN $ids "
                 "RETURN d.document_id AS doc, c.chunk_id AS id, c.text AS text, c.pdf_page AS page",
                 ids=PENALTY_CHUNK_IDS + [PAGE94_CHUNK_ID]).data()
driver.close()
same = [r["id"] for r in rows if r["text"] == local[r["id"]]["text"] and r["page"] == local[r["id"]]["pdf_page"]]
print("94쪽 1개 + 벌칙 12개 =", len(PENALTY_CHUNK_IDS) + 1, "| DB에 있는 것", len(rows), "| 로컬 블록과 본문·쪽 일치", len(same))

Chunk 1743 | Document 10 | HAS_CHUNK 1743


94쪽 1개 + 벌칙 12개 = 13 | DB에 있는 것 13 | 로컬 블록과 본문·쪽 일치 13


## 4. 벌칙 조문이 빠져 있던 이유 (승우님 요청 확인)

- 법 문서 선정 쪽이 `6 7 18 19쪽`(제2조, 제11~13조)뿐이었다. 벌칙(제77~83조)은 **처음부터 적재 범위 밖**이었다. 적재 실패나 누락 사고가 아니다.
- 골든 Q28을 "근거 부족" 질문으로 설계하면서 제외했다. 위치는 법 PDF 55~58쪽이고, 진단·신고기준 지침 27쪽 "3.5 신고의무 위반에 따른 벌칙"에도 같은 요약이 있다.
- **특이점(겹친 판):** 법 57·58쪽에는 현행 조문(2026.10.8 시행본)과 개정 예정본(`삭제<2026. 10. 20.>`, 2027.10.21 시행)의 제81조·제83조가 연달아 실려 있다. 프로젝트 기준(현행본)에 맞춰 **첫 번째 판만** 잘랐고 예정본은 넣지 않았다.
- 신고 지연과 직접 관련된 조문: 제79조의4(제1·2급 신고 위반, 500만원 이하), 제80조 제1·2호(제3·4급, 300만원 이하), 제81조 제3·4호(제12조 ① 신고를 게을리함, 200만원 이하).
- **알려진 한계:** 제83조 ⑤ 끝의 개정 이력 표시와 제81조의2의 신설 표시, 2건이 블록 본문에 없다(처벌 내용에는 영향 없는 주석). 벌칙 질문에 답하는 **카드는 만들지 않았다**(적재만). 법 제79조·제80조는 쪽을 넘어 형량과 적용 대상이 서로 다른 블록에 있다.

## 5. UI 변경 요약 (`src/rag/ui/`, 주승우님 파일 — 승우님 확인 필요)

| 항목 | 일반 사용자 | 개발자(`?dev=1`) |
|---|---|---|
| 화면 | 질문 하나로 통합(예시 버튼·지역 선택 제거), 사이드바 없음 | + 참고 문서 허용 목록, 답변 다듬기 켜기/끄기, 도구 한계 |
| 답변 | 상태 + 최종 답변 + **근거 문서(문서명·판본·PDF 쪽수)** + 확인이 필요한 부분 | + 원문 정리형, 출처 발췌, 처리 흐름, 그래프 경로, 대시보드 |
| 저장한 답변 | 세션 안에서만 유지, 삭제, `.md` 내려받기(근거 문서·안내 포함, 질문 원문 포함 안내) | 같음 |

- 개발자 화면은 `DEV_PASSWORD`가 맞을 때만 열린다(비어 있으면 닫힘, 로컬은 `DEV_ALLOW_OPEN=1`). Codex 검토 지적으로 **기본값을 닫힘으로 바꿨다**.
- 질문 원문은 이력(`data/interim/ui_history/`, Git 제외)에 저장되고, 사용자 화면에 저장·개인정보 입력 주의 안내를 넣었다.
- **철회:** "확인할 곳"(문의처 안내)은 사용자 결정으로 만들었다가 뺐다. 근거 부족 안내에서도 기관 언급을 삭제했다.
- 검증은 Streamlit `AppTest`(렌더링·질문 실행·저장)까지이며 **브라우저 시각 확인은 하지 않았다**.

### 원문 출처 링크: 쪽수가 아니라 표지·개정이력·목차로 대조
공식 PDF를 내려받아 우리 원본과 앞 14쪽(표지·개정 이력·목차)의 본문, 개정·발간일 줄을 비교했다. 네 문서 모두 유사도 1.0000이었다.
수족구 PDF는 표지가 "2025년도"로 적혀 있으나 지침 연혁표 끝이 "2026년도 [제10판] 2026. 1."이고 우리 원본과 같은 파일이다(다른 주소의 2025년판 파일은 사용하지 않음).

In [5]:
# 읽기 전용: manifest 에 채운 공식 URL 을 읽기만 한다(URL 에 접속하지 않는다).
from rag.ui import sources
for doc_id, url in sources.load_urls().items():
    print(f"{doc_id:38s} {url}")

kdca_measles_2026                      https://kdca.go.kr/bbs/kdca/55/311031/artclView.do
kdca_diagnosis_reporting_2026          https://www.kdca.go.kr/bbs/kdca/55/311073/artclView.do
kdca_respiratory_2026                  https://kdca.go.kr/bbs/kdca/55/304986/download.do
kdca_flu_facility_2026                 https://kdca.go.kr/bbs/kdca/55/310106/download.do
kdca_hfmd_2026                         https://www.kdca.go.kr/bbs/kdca/55/311285/artclView.do
kdca_vaccine_preventable_2026          https://www.kdca.go.kr/bbs/kdca/55/311032/artclView.do


## 6. Codex 검토에서 고친 것 (특이점)

- 법 제12조 ① 카드(S-L01·Q09): 법의 적용 범위(제1급~제3급, 복지부령)와 주체(제2호) 한정 누락 → 수정. 같은 근거를 쓰는 **Q10 카드와 샘플 경로의 제12조 카드**도 같은 인용·주장으로 맞춤(샘플 카드는 인용문에 없는 '홍역 포함'을 주장에 담고 있었음).
- 개발자 화면 기본 공개 → 비밀번호가 비면 닫힘. CSV 수식 주입 방지, 비정상 이력 줄 건너뛰기, 마크다운·URL 이스케이프, 내부 문구(도구명·DB 대조) 사용자 화면에서 제외.
- 저장·내려받기 파일에 일부 확인 안내 포함, 질문 원문 포함 안내.
- 적재 함수: 청크 ID 접두사와 문서 ID 일치, 두 문서가 정확히 하나씩인지 검사 보강. 오래된 주석("블록 파일은 33개", "고정 샘플만") 수정.
- 기록 정정: 진행 문서의 "HAS_CHUNK 영향 없음"은 틀렸고(1730→1731) 바로잡음.

## 7. 한계와 남은 일

- **독립 2/8:** 현재 Tool은 사람이 만든 카드가 있는 질문에만 근거를 붙여 답한다. 근거가 DB에 있는 독립 6개(I1~I6)는 모두 답하지 못했다. 이 한계는 독립성 보존과 11/8 동결 일정 때문에 의도적으로 고치지 않았다.
- 11/8: PDF 원문 쪽·문구 확인(23·19·93·94·116쪽), 최종 재실행·검증표, 백업.
- 확인 못 한 것: 진단·신고기준 고시 번호(지침 2026-4호 vs 법령 게시판 2026-12호), 감염병예방법 시행본 연혁 주소, UI 브라우저 시각 확인.
- 팀 협의 대기: 공동 3개 필수 시연, 성홍열 2-Graph(T4), 3인 온톨로지 승인 기록, a05·a06 개념 의미, 테스트 격리 합의.
- 승우님 쪽: S-L01 기대 상태를 PARTIAL로 조정, 벌칙 질문은 카드가 없어 근거 부족임을 반영.